# 第 8 课：把所有内容串起来 - 训练 Wordle

<div style="background-color:#fff6ff; padding:13px; border-width:3px; border-color:#efe6ef; border-style:solid; border-radius:6px">
<p> 💻 &nbsp; <b>访问 <code>requirements.txt</code> 文件：</b> 1) 点击 notebook 顶部菜单中的 <em>"File"</em>，然后 2) 点击 <em>"Open"</em>。

<p> ⬇ &nbsp; <b>下载 Notebook：</b> 1) 点击 notebook 顶部菜单中的 <em>"File"</em>，然后 2) 点击 <em>"Download as"</em> 并选择 <em>"Notebook (.ipynb)"</em>。</p>

<p> 📒 &nbsp; 如需更多帮助，请参阅 <em>"Appendix – Tips, Help, and Download"</em> 课程。</p>

</div>

导入依赖并设置用于训练的 Predibase deployment：

In [1]:
import os

from predibase import (
    Predibase,
    GRPOConfig,
    RewardFunctionsConfig,
    RewardFunctionsRuntimeConfig,
    SFTConfig,
    SamplingParamsConfig,
)
from datasets import load_dataset
from dotenv import load_dotenv

In [2]:
load_dotenv("../.env")
pb = Predibase(api_token=os.environ["PREDIBASE_API_KEY"])

The `PREDIBASE_API_TOKEN` long format you're using will be deprecated on April 15, 2024. Please upgrade your token by going to the Predibase UI and generating a new one.


WARN: Currently installed SDK is outdated. This can lead to bugs or unexpected behavior. Consider upgrading to the 
latest version. Installed: 2025.4.1 Latest: 2025.12.1.

WARN: Currently installed SDK is outdated. This can lead to bugs or unexpected behavior. Consider upgrading to the 
latest version. Installed: 2025.4.1 Latest: 2025.12.1.

Connected to Predibase as User(id=b9aa2fa4-f9fa-48a2-8a04-be5a301e63f6, username=support+dlai@predibase.com)

从 Hugging Face 加载 GRPO [wordle training dataset](https://huggingface.co/datasets/predibase/wordle-grpo)：

In [ ]:
# 从 Hugging Face 加载数据集
dataset = load_dataset("predibase/wordle-grpo", split="train")
dataset = dataset.to_pandas()

# 将数据集上传到 Predibase
try:
    dataset = pb.datasets.from_pandas_dataframe(
        dataset,
        name="wordle_grpo_data"
    )
except Exception:
    dataset = pb.datasets.get("wordle_grpo_data")

README.md:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

train.csv: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/76 [00:00<?, ? examples/s]

WARN: Currently installed SDK is outdated. This can lead to bugs or unexpected behavior. Consider upgrading to the 
latest version. Installed: 2025.4.1 Latest: 2025.12.1.

/usr/local/lib/python3.11/site-packages/predibase/_errors.py:53: UserWarning: Currently installed SDK is outdated. This can lead to bugs or unexpected behavior. Consider upgrading to the latest version. Installed: 2025.4.1 Latest: 2025.12.1.
  warn(


创建 training repo 并加载 Wordle 奖励函数：

In [ ]:
# 如果你在自己的环境中运行，请取消下面一行的注释；这里已经为你设置好了 repo
# 在 Predibase 中创建仓库
# repo = pb.repos.create(name="wordle", exists_ok=True)

<div style="background-color:#fff6ff; padding:13px; border-width:3px; border-color:#efe6ef; border-style:solid; border-radius:6px">
<b>注意：</b> 你可以通过 1) 点击 notebook 顶部菜单中的 <em>"File"</em>，然后 2) 点击 <em>"Open"</em>，来查看存放在 <code>reward_functions.py</code> 中的奖励函数完整代码。

</div>

In [ ]:
# 导入奖励函数
from reward_functions import (
    guess_value,
    output_format_check,
    uses_previous_feedback,
)

## 设置训练任务

<div style="background-color:#fff6ff; padding:13px; border-width:3px; border-color:#efe6ef; border-style:solid; border-radius:6px">
<b>注意：</b> 下方单元在学习平台上无法运行。如果你决定在自己的电脑上运行，请在上面的设置中把环境变量 PREDIBASE_API_TOKEN 更新为你自己的 API key。

你可以在 [this website](https://predibase.com/free-trial) 获取免费额度来试用 Predibase。

</div>

In [ ]:
# 在 Predibase 中创建 GRPO 训练任务，并指定配置、
# 数据集、仓库以及奖励函数
pb.finetuning.jobs.create(
    config=GRPOConfig(
        base_model="qwen2-5-7b-instruct",
        reward_fns=RewardFunctionsConfig(
            runtime=RewardFunctionsRuntimeConfig(
                packages=["pandas"]
            ),
            functions={
                "output_format_check": output_format_check,
                "uses_previous_feedback": uses_previous_feedback,
                "guess_value": guess_value,
            }
        ),
        sampling_params=SamplingParamsConfig(max_tokens=4096),
        num_generations=16
    ),
    dataset=dataset,
    repo="wordle",
    description="Wordle GRPO"
    )

RuntimeError: Bad request. Response status code 403. Error: {'message': 'You have insufficient permissions to perform this action.'}

## 在 Predibase 上试用 SFT 和 SFT+GRPO

你可以使用下面的代码在 Predibase 上设置一个 SFT 训练任务，然后将得到的 checkpoint 作为 GRPO 训练的输入。

这个示例使用的是 Hugging Face 上提供的 [Wordle SFT dataset](https://huggingface.co/datasets/predibase/wordle-sft)。

### 在 Predibase 上进行 SFT 训练

```python

dataset = load_dataset("predibase/wordle-sft", split="train")
dataset = dataset.to_pandas()

# 将数据集上传到 Predibase
dataset = pb.datasets.from_pandas_dataframe(dataset, name="wordle_sft_data")

# 在 Predibase 中创建仓库
repo = pb.repos.create(name="wordle", exists_ok=True)

# 在 Predibase 中创建 SFT 训练任务，并指定配置、数据集、仓库和奖励函数
pb.finetuning.jobs.create(
    config=SFTConfig(
        base_model="qwen2-5-7b-instruct",
        epochs=10,
        rank=64,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "down_proj", "up_proj"],
    ),
    dataset=dataset,
    repo="wordle",
    description="Wordle SFT, 10 epochs"
 )
```

### 在 Predibase 上进行 SFT + GRPO 训练

```python
# 使用与 GRPO 训练任务相同的数据集
dataset = pb.datasets.get("wordle_grpo_data")

# 在 Predibase 中创建 GRPO 训练任务，并指定配置、数据集、仓库和奖励函数
pb.finetuning.jobs.create(
    config=GRPOConfig(
        base_model="qwen2-5-7b-instruct",
        reward_fns=RewardFunctionsConfig(
            runtime=RewardFunctionsRuntimeConfig(packages=["pandas"]),
            functions={
                "output_format_check": output_format_check,
                "uses_previous_feedback": uses_previous_feedback,
                "guess_value": guess_value,
            }
        ),
        epochs=3,
        enable_early_stopping=False,
        sampling_params=SamplingParamsConfig(max_tokens=4096),
        num_generations=8
    ),
    continue_from_version="wordle/1", # 把 "1" 改成仓库中 SFT 训练任务对应的版本号
    dataset=dataset,
    repo="wordle",
    description="Wordle GRPO"
 )
```